# Crop Disease Detection — EfficientNet-B0 Training

**Runtime**: GPU (T4 recommended) — Runtime → Change runtime type → T4 GPU  
**Dataset**: PlantVillage (38 classes, ~54k images) — preprocessed splits in S3
**Model**: EfficientNet-B0 fine-tuned with PyTorch  
**Tracking**: MLflow (remote server on EC2)  

## Workflow
1. Install dependencies
2. Download from S3
3. Build DataLoaders with augmentation
4. Fine-tune EfficientNet-B0
5. Evaluate on test set
6. Export to ONNX
7. Log everything to MLflow and register model

## 0. Setup

In [1]:
# Install dependencies not available in Colab by default
!pip install mlflow boto3 --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 51.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.5/140.5 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.6/8

In [ ]:
import io
import os
import json
import time
import boto3
import pandas as pd
from pathlib import Path

import onnx
import mlflow
import mlflow.pytorch
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import classification_report, f1_score
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.datasets import ImageFolder
from torchvision.models import EfficientNet_B0_Weights, efficientnet_b0

# verify GPU
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

Device: cuda
GPU: Tesla T4


## 1. Configuration

Edit the values in this cell before running.

In [ ]:
# ── MLflow ────────────────────────────────────────────────────────────────────
MLFLOW_TRACKING_URI = "http://your-ec2-dns:5000"  # <- set your EC2 URI
EXPERIMENT_NAME = "crop-disease-detection"
MODEL_NAME = "crop-disease-efficientnet-b0"  # name in MLflow Model Registry

# ── Data ──────────────────────────────────────────────────────────────────────
# S3
S3_BUCKET = "your_model_bucket_name"
S3_DATA_PREFIX = "data/processed"
LOCAL_DATA_DIR = Path("/content/data/processed")

# ── Training hyperparameters ───────────────────────────────────────────────────
BATCH_SIZE = 64
NUM_EPOCHS = 15
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_PATIENCE = 3       # reduce LR if val loss doesn't improve for N epochs
EARLY_STOP_PATIENCE = 6
NUM_WORKERS = 2
RANDOM_SEED = 42

# ── Image ─────────────────────────────────────────────────────────────────────
IMAGE_SIZE = 224      # EfficientNet-B0 standard input
# ImageNet normalisation — used because EfficientNet-B0 was pre-trained on ImageNet
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

In [ ]:
# test that mlflow works
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
print(mlflow.get_tracking_uri())
client = mlflow.tracking.MlflowClient()
print(client.search_experiments())  # should return empty list or existing experiments

http://ec2-52-211-42-124.eu-west-1.compute.amazonaws.com:5000
[<Experiment: artifact_location='s3://crop-disease-models-mlflow-stg-478544568263/0', creation_time=1781087074363, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1781087074363, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]


## 2. Load Data

In [ ]:
# install AWS CLI on this Colab instance
!pip install awscli --quiet

In [ ]:
# ── Download from S3 ────────────────────────────────────────────────
import boto3
from google.colab import userdata

# store AWS keys as Colab secrets (🔑 icon in the left sidebar)
os.environ["AWS_ACCESS_KEY_ID"]     = userdata.get("AWS_ACCESS_KEY_ID")
os.environ["AWS_SECRET_ACCESS_KEY"] = userdata.get("AWS_SECRET_ACCESS_KEY")
os.environ["AWS_DEFAULT_REGION"]    = "eu-west-1"

LOCAL_DATA_DIR.mkdir(parents=True, exist_ok=True)
print(f"Downloading from s3://{S3_BUCKET}/{S3_DATA_PREFIX}/...")
!aws s3 sync s3://{S3_BUCKET}/{S3_DATA_PREFIX}/ {LOCAL_DATA_DIR}/
DATA_DIR = LOCAL_DATA_DIR

# load class metadata saved by preprocess.py
with open(DATA_DIR / "metadata.json") as f:
    metadata = json.load(f)

CLASS_NAMES = metadata["class_names"]
NUM_CLASSES = metadata["num_classes"]
print(f"Classes: {NUM_CLASSES}")
print(f"First 5: {CLASS_NAMES[:5]}")

## 3. DataLoaders

Why `torchvision.datasets.ImageFolder`?  
It reads a directory structured as `split/class_name/*.jpg` — exactly what `preprocess.py` produces — and returns (tensor, label_index) pairs ready for DataLoader. We add augmentation only on the training split.

In [8]:
# Training transforms: resize + augmentation + normalize
# Augmentation teaches the model to be robust to real-world variation
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

# Val/test transforms: only resize + normalize (no augmentation)
eval_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

train_dataset = ImageFolder(root=str(DATA_DIR / "train"), transform=train_transforms)
val_dataset   = ImageFolder(root=str(DATA_DIR / "val"),   transform=eval_transforms)
test_dataset  = ImageFolder(root=str(DATA_DIR / "test"),  transform=eval_transforms)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset)} images | Val: {len(val_dataset)} | Test: {len(test_dataset)}")

# sanity check: class order from ImageFolder must match metadata
assert train_dataset.classes == CLASS_NAMES, \
    f"Class mismatch between ImageFolder and metadata.json!\n{train_dataset.classes[:3]} vs {CLASS_NAMES[:3]}"

Train: 37997 images | Val: 8145 | Test: 8163


## 4. Model — EfficientNet-B0

Transfer learning strategy:
- Load EfficientNet-B0 with **ImageNet pre-trained weights**
- **Freeze** all layers initially — only train the new classification head
- After a few epochs, **unfreeze** all layers for full fine-tuning

In [9]:
def build_model(num_classes: int, freeze_backbone: bool = True) -> nn.Module:
    """
    Load EfficientNet-B0 with ImageNet weights and replace the
    classifier head for our 38-class problem.
    """
    model = efficientnet_b0(weights=EfficientNet_B0_Weights.DEFAULT)

    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False

    # replace the classifier head
    # original: Linear(1280, 1000) for ImageNet
    # ours:     Linear(1280, 38)   for PlantVillage
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, num_classes),
    )

    return model.to(DEVICE)


model = build_model(NUM_CLASSES, freeze_backbone=True)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Total params:     {sum(p.numel() for p in model.parameters()):,}")

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 56.6MB/s]


Trainable params: 48,678
Total params:     4,056,226


## 5. Training Loop

In [10]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += images.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        outputs = model(images)
        loss = criterion(outputs, labels)
        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += images.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())
    f1 = f1_score(all_labels, all_preds, average="weighted")
    return total_loss / total, correct / total, f1

## 6. MLflow Training Run

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

with mlflow.start_run() as run:
    RUN_ID = run.info.run_id
    print(f"MLflow run ID: {RUN_ID}")

    # ── log hyperparameters ───────────────────────────────────────────────────
    mlflow.log_params({
        "model": "efficientnet_b0",
        "pretrained": "imagenet",
        "num_classes": NUM_CLASSES,
        "batch_size": BATCH_SIZE,
        "num_epochs": NUM_EPOCHS,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "image_size": IMAGE_SIZE,
        "optimizer": "AdamW",
        "scheduler": "ReduceLROnPlateau",
        "early_stop_patience": EARLY_STOP_PATIENCE,
        "train_size": len(train_dataset),
        "val_size": len(val_dataset),
        "test_size": len(test_dataset),
        "leaf_aware_split": metadata.get("leaf_aware_split", False),
    })

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", patience=LR_PATIENCE, factor=0.5
    )

    best_val_loss = float("inf")
    best_val_f1 = 0.0
    epochs_no_improve = 0
    unfreeze_done = False

    for epoch in range(1, NUM_EPOCHS + 1):
        # ── unfreeze backbone after epoch 3: you can comment out this if statement if you want to train more ───
        #if epoch == 4 and not unfreeze_done:
        #    print("Unfreezing backbone for full fine-tuning...")
        #    for param in model.parameters():
        #        param.requires_grad = True
        #    optimizer = optim.AdamW(
        #        model.parameters(),
        #        lr=LEARNING_RATE / 10,  # lower LR for fine-tuning
        #        weight_decay=WEIGHT_DECAY,
        #    )
        #    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        #        optimizer, mode="min", patience=LR_PATIENCE, factor=0.5
        #    )
        #    unfreeze_done = True

        t0 = time.time()
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, val_f1 = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0

        scheduler.step(val_loss)

        print(
            f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
            f"train loss {train_loss:.4f} acc {train_acc:.4f} | "
            f"val loss {val_loss:.4f} acc {val_acc:.4f} f1 {val_f1:.4f} | "
            f"{elapsed:.1f}s"
        )

        # ── log metrics to MLflow ─────────────────────────────────────────────
        mlflow.log_metrics({
            "train_loss": train_loss,
            "train_acc": train_acc,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
            "lr": optimizer.param_groups[0]["lr"],
        }, step=epoch)

        # ── save best model ───────────────────────────────────────────────────
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_val_f1 = val_f1
            epochs_no_improve = 0
            torch.save(model.state_dict(), "/content/best_model.pt")
        else:
            epochs_no_improve += 1

        # ── early stopping ────────────────────────────────────────────────────
        if epochs_no_improve >= EARLY_STOP_PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"\nBest val loss: {best_val_loss:.4f} | Best val F1: {best_val_f1:.4f}")

MLflow run ID: 31c978e10c9e440c8a82f882b6b65e9f
Epoch 01/15 | train loss 2.2728 acc 0.5464 | val loss 1.4387 acc 0.7684 f1 0.7422 | 191.0s
Epoch 02/15 | train loss 1.2405 acc 0.7820 | val loss 0.8717 acc 0.8570 f1 0.8451 | 192.4s
Epoch 03/15 | train loss 0.8732 acc 0.8374 | val loss 0.6357 acc 0.8861 f1 0.8800 | 191.0s
Epoch 04/15 | train loss 0.6917 acc 0.8640 | val loss 0.5131 acc 0.9014 f1 0.8982 | 189.4s
Epoch 05/15 | train loss 0.5878 acc 0.8764 | val loss 0.4310 acc 0.9118 f1 0.9099 | 190.1s
Epoch 06/15 | train loss 0.5190 acc 0.8858 | val loss 0.3811 acc 0.9198 f1 0.9186 | 194.3s
Epoch 07/15 | train loss 0.4664 acc 0.8934 | val loss 0.3420 acc 0.9222 f1 0.9210 | 194.7s
Epoch 08/15 | train loss 0.4302 acc 0.8974 | val loss 0.3154 acc 0.9263 f1 0.9250 | 189.0s
Epoch 09/15 | train loss 0.4024 acc 0.9035 | val loss 0.2894 acc 0.9310 f1 0.9302 | 193.8s
Epoch 10/15 | train loss 0.3737 acc 0.9078 | val loss 0.2746 acc 0.9331 f1 0.9323 | 189.4s
Epoch 11/15 | train loss 0.3578 acc 0.9103

## 7. Evaluate on Test Set

In [14]:
# load best checkpoint and evaluate on held-out test set
model.load_state_dict(torch.load("/content/best_model.pt", map_location=DEVICE))

with mlflow.start_run(run_id=RUN_ID):
    test_loss, test_acc, test_f1 = evaluate(model, test_loader, criterion)
    print(f"Test accuracy: {test_acc:.4f} | Test F1: {test_f1:.4f}")

    mlflow.log_metrics({
        "test_loss": test_loss,
        "test_acc": test_acc,
        "test_f1": test_f1,
    })

    # full per-class classification report
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for images, labels in test_loader:
            outputs = model(images.to(DEVICE))
            all_preds.extend(outputs.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    report = classification_report(all_labels, all_preds, target_names=CLASS_NAMES)
    print(report)

    # log report as artifact
    with open("/content/classification_report.txt", "w") as f:
        f.write(report)
    mlflow.log_artifact("/content/classification_report.txt")

Test accuracy: 0.9462 | Test F1: 0.9457
                                                    precision    recall  f1-score   support

                                Apple___Apple_scab       0.91      0.85      0.88        95
                                 Apple___Black_rot       1.00      0.98      0.99        94
                          Apple___Cedar_apple_rust       1.00      0.98      0.99        42
                                   Apple___healthy       0.96      0.95      0.95       247
                               Blueberry___healthy       0.96      1.00      0.98       226
          Cherry_(including_sour)___Powdery_mildew       0.99      0.96      0.97       158
                 Cherry_(including_sour)___healthy       0.98      0.98      0.98       129
Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot       0.90      0.73      0.81        77
                       Corn_(maize)___Common_rust_       0.98      0.97      0.97       179
               Corn_(maize)___Northern_

## 8. Export to ONNX

Why ONNX?
- The inference service (FastAPI / Lambda) uses **ONNX Runtime** instead of PyTorch
- ONNX Runtime is ~100MB vs PyTorch ~2GB — critical for Lambda container size limits
- Same model, same predictions, much smaller footprint

In [ ]:
# intsall onnxscript on this Colab instance
!pip install onnxscript --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 714.8/714.8 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.6/17.6 MB 87.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 166.8/166.8 kB 17.0 MB/s eta 0:00:00


In [18]:
ONNX_PATH = "/content/model.onnx"

model.eval()
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(DEVICE)

torch.onnx.export(
    model,
    dummy_input,
    ONNX_PATH,
    opset_version=18,
    input_names=["image"],
    output_names=["logits"],
    dynamic_axes={
        "image":  {0: "batch_size"},
        "logits": {0: "batch_size"},
    },
)

print(f"ONNX model exported → {ONNX_PATH}")

# quick sanity check with onnxruntime
!pip install onnxruntime --quiet
import onnxruntime as ort

session = ort.InferenceSession(ONNX_PATH)
dummy_np = dummy_input.cpu().numpy()
onnx_out = session.run(["logits"], {"image": dummy_np})[0]
print(f"ONNX output shape: {onnx_out.shape}")  # should be (1, 38)

# verify outputs match PyTorch
with torch.no_grad():
    pt_out = model(dummy_input).cpu().numpy()
np.testing.assert_allclose(onnx_out, pt_out, rtol=1e-3, atol=1e-5)
print("PyTorch and ONNX outputs match ✓")

/tmp/ipykernel_1414/2655911989.py:6: UserWarning: # 'dynamic_axes' is not recommended when dynamo=True, and may lead to 'torch._dynamo.exc.UserError: Constraints violated.' Supply the 'dynamic_shapes' argument instead if export is unsuccessful.
  torch.onnx.export(


[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`...
[torch.onnx] Obtain model graph for `EfficientNet([...]` with `torch.export.export(..., strict=False)`... ✅
[torch.onnx] Run decompositions...


/usr/lib/python3.12/copyreg.py:99: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


[torch.onnx] Run decompositions... ✅
[torch.onnx] Translate the graph into ONNX...
[torch.onnx] Translate the graph into ONNX... ✅
[torch.onnx] Optimize the ONNX graph...
[torch.onnx] Optimize the ONNX graph... ✅
ONNX model exported → /content/model.onnx
ONNX output shape: (1, 38)
PyTorch and ONNX outputs match ✓


## 9. Log Model to MLflow and Register

In [ ]:
## 10. Generate Reference Data for Evidently Monitoring
#
# Runs the trained model on the full test set and saves the prediction
# distribution to S3 as a Parquet file.
#
# This file becomes the REFERENCE DATA for drift monitoring:
#   - Reference: these test-set predictions (stable, computed once)
#   - Current:   recent API predictions from PostgreSQL (updated daily)
# Evidently compares the two to detect distribution shift.

REFERENCE_S3_KEY = "monitoring/reference_predictions.parquet"

model.load_state_dict(torch.load("/content/best_model.pt", map_location=DEVICE))
model.eval()

records = []
with torch.no_grad():
    for images, _ in test_loader:
        probs_batch = torch.softmax(model(images.to(DEVICE)), dim=1).cpu().numpy()
        for probs in probs_batch:
            top_idx = int(probs.argmax())
            entropy = float(-np.sum(probs * np.log(probs + 1e-9)))
            records.append({
                "predicted_class": CLASS_NAMES[top_idx],
                "confidence": float(probs[top_idx]),
                "entropy": entropy,
            })

reference_df = pd.DataFrame(records)
print(f"Reference data: {len(reference_df)} predictions")
print(f"Unique classes: {reference_df['predicted_class'].nunique()}")
print("\nClass distribution (top 10):")
print(reference_df['predicted_class'].value_counts().head(10))
print(f"\nConfidence stats:")
print(reference_df['confidence'].describe())

# ── upload to S3 ──────────────────────────────────────────────────────────
buf = io.BytesIO()
reference_df.to_parquet(buf, index=False)
buf.seek(0)

boto3.client("s3").put_object(
    Bucket=S3_BUCKET,
    Key=REFERENCE_S3_KEY,
    Body=buf.getvalue(),
)
print(f"\nReference data uploaded → s3://{S3_BUCKET}/{REFERENCE_S3_KEY}")

# ── log to MLflow as artifact ─────────────────────────────────────────────
with mlflow.start_run(run_id=RUN_ID):
    with open("/content/reference_predictions.parquet", "wb") as f:
        buf.seek(0)
        f.write(buf.read())
    mlflow.log_artifact("/content/reference_predictions.parquet", artifact_path="monitoring")
    print(f"Reference data also logged to MLflow run {RUN_ID}")

In [ ]:
with mlflow.start_run(run_id=RUN_ID):

    # log metadata.json so the inference service knows class names
    mlflow.log_artifact(str(DATA_DIR / "metadata.json"), artifact_path="onnx")

    # 1. Properly log the ONNX model as an official MLflow Model flavor
    # This automatically generates the MLmodel file
    onnx_model = onnx.load(ONNX_PATH)
    
    mlflow.onnx.log_model(
        onnx_model=onnx_model,
        artifact_path="onnx",
    )

    # register the PyTorch model in the Model Registry
    # (registered separately from ONNX for experiment comparison)
    model_info = mlflow.pytorch.log_model(
        pytorch_model=model,
        artifact_path="model",
        registered_model_name=MODEL_NAME,
    )

print(f"\nModel registered as: {MODEL_NAME}")
print(f"Run ID: {RUN_ID}")
print(f"\nNext steps:")
print(f"  1. Go to MLflow UI → Models → {MODEL_NAME}")
print(f"  2. Promote version to 'Staging', then 'Production'")
print(f"  3. Set RUN_ID={RUN_ID} in Lambda env vars or SSM Parameter Store")

2026/06/10 11:39:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/10 11:39:24 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is to set `serialization_format` to 'pt2' to save the PyTorch model using the safe graph model format.
2026/06/10 11:39:24 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
2026/06/10 11:39:37 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version

🏃 View run polite-snake-541 at: http://ec2-52-211-42-124.eu-west-1.compute.amazonaws.com:5000/#/experiments/1/runs/31c978e10c9e440c8a82f882b6b65e9f
🧪 View experiment at: http://ec2-52-211-42-124.eu-west-1.compute.amazonaws.com:5000/#/experiments/1

Model registered as: crop-disease-efficientnet-b0
Run ID: 31c978e10c9e440c8a82f882b6b65e9f

Next steps:
  1. Go to MLflow UI → Models → crop-disease-efficientnet-b0
  2. Promote version to 'Staging', then 'Production'
  3. Set RUN_ID=31c978e10c9e440c8a82f882b6b65e9f in Lambda env vars or SSM Parameter Store


## 10. Promote to Production (after manual review)

In [ ]:
# Run this cell only after reviewing the metrics and deciding to promote

client = mlflow.tracking.MlflowClient()

# get the latest version
versions = client.search_model_versions(f"name='{MODEL_NAME}'")
latest_version = sorted(versions, key=lambda v: int(v.version))[-1].version

# transition to Staging first
client.set_registered_model_alias(
    name=MODEL_NAME,
    version=latest_version,
    alias="Staging",
)
print(f"Version {latest_version} → Staging")


# then to Production (only after validation)
client.set_registered_model_alias(
    name=MODEL_NAME,
    version=latest_version,
    alias="Production",
)
print(f"Version {latest_version} → Production")
print(f"\nexport RUN_ID={RUN_ID}")